# 03 · A small machine-learning workflow
Question: can petal length and width help distinguish **versicolor** from **virginica**?

Goals: define features and a target, split data, compare a fitted model with a baseline, and connect pandas tables to NumPy calculations.

This notebook reloads its own data and can run independently. The full-data exploration in notebook 02 is a separate teaching exercise. Here the feature choice is fixed in advance, and we reserve a validation set before fitting.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss

iris = load_iris(as_frame=True)
df = iris.frame.loc[iris.frame["target"].isin([1, 2])].copy()
features = ["petal length (cm)", "petal width (cm)"]
X = df[features].to_numpy()
y = (df["target"] == 2).astype(int).to_numpy()
print("X:", X.shape, "y:", y.shape)
print("0 = versicolor, 1 = virginica")
assert X.shape == (100, 2) and y.shape == (100,)

## A. Training and validation
Training data fits parameters. Validation data lets us evaluate or compare choices. For a final report after model selection, reserve a separate test set.

The seed makes this split repeatable. Stratification keeps the class proportions. A random split suits this toy illustration; data with repeated people, groups or time ordering need a split that respects that structure.

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Validation:", X_valid.shape)
assert len(X_train) + len(X_valid) == 100

## B. A baseline and a fitted model
The baseline always predicts the most frequent training class. Logistic regression uses the feature values to estimate a probability.

The pipeline learns scaling parameters from training data only. It applies those same parameters to validation data. Fitting the scaler before splitting would leak information.

We use scikit-learn to keep attention on the workflow.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)

baseline_prediction = baseline.predict(X_valid)
prediction = model.predict(X_valid)
probability = model.predict_proba(X_valid)[:, 1]
results = pd.DataFrame({
    "model": ["Majority baseline", "Logistic regression"],
    "validation accuracy": [accuracy_score(y_valid, baseline_prediction),
                            accuracy_score(y_valid, prediction)],
})
print(results.to_string(index=False))
print("Mean validation log loss:", log_loss(y_valid, probability))

## C. The NumPy calculation behind a probability
`X @ w + b` produces a score for each row. The sigmoid turns a score into a probability.

Use the **scaled** features because the fitted weights correspond to that coordinate system. The demonstration scores are moderate, so the direct exponential expression is suitable here. More extreme inputs require a numerically stable sigmoid.

In [ ]:
scaler, classifier = model.steps[0][1], model.steps[1][1]
X_scaled = scaler.transform(X_valid)
w = classifier.coef_[0]
b = classifier.intercept_[0]
scores = X_scaled @ w + b
probability_numpy = 1 / (1 + np.exp(-scores))
print("Shapes:", X_scaled.shape, w.shape, scores.shape)
np.testing.assert_allclose(probability_numpy, probability)
print("NumPy and scikit-learn probabilities agree.")

### Exercise G · Inspect errors (6 min)
1. Build a pandas table with the true class, prediction and predicted probability of virginica.
2. Filter the rows where the prediction is wrong.
3. Explain one error: was the probability close to 0.5 or was the model confident?

Checkpoint: the table has 25 rows. The number of errors must match the accuracy calculation. A high score on 25 flowers is limited evidence about other populations.

Optional: try one feature, then compare validation results. Repeatedly using this set to choose features makes it part of model selection. It cannot then serve as an untouched final test.

In [ ]:
# Build the result table and inspect mistakes here.

## Exit ticket
What are one row of `X` and one entry of `y`? Why do we compare to a baseline? Which data fitted the scaler? What would need to be saved so another person could reproduce this run?

## Sources
[scikit-learn pitfalls and pipelines](https://scikit-learn.org/stable/common_pitfalls.html), [Iris dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html). Reading: *An Introduction to Statistical Learning*, Statistical Learning and Classification; *Python Data Science Handbook*, Machine Learning.